# 07. 결함 요인 모델 (D66, 2026-11-24)

- 목적: **판정이 아니라 요인 분석**. 어떤 질문·상품 조건에서 결함이 많이 생기는지 본다. 판정은 규칙(03)이 한다.
- 라벨 (`outputs/judgments.csv`): A = 기준③ 처리 '제외', B = 기준④ '기재 없음'.
- 피처: `docs/reference/결함요인모델_참고구현.py`와 같다. 규칙 누수를 막기 위해 고객 성향·상품 등급·cot2 허용 범위는 넣지 않는다.
- 학습 Training / 평가 Validation. 로지스틱 회귀(class_weight balanced) + 의사결정나무(깊이 3, min_samples_leaf 50).
- 함수: `src/factor_model.py`. 결과: `outputs/model/` (원문 없음). 숫자만 둔다. 해석은 쓰지 않는다.
- 노트북 번호 07은 D66으로 다시 정의했다. 기존 `07_근거검색_LLM진단`은 D53의 근거 검색 단계로 그대로 둔다.

데이터: AI Hub(한국지능정보사회진흥원) 사업 결과

In [ ]:
import sys
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
pd.set_option("display.width", 220)
from src.load_cot import load_cot, load_products
from src.judge import judge
from src.factor_model import build_features, fit_models, factor_rates, LABELS
AS_OF = "2026-11-24"
cot = load_cot(); prod = load_products()
_, ctx = judge(cot, prod, as_of=AS_OF)
jd = pd.read_csv(ROOT / "outputs" / "judgments.csv", encoding="utf-8-sig", keep_default_na=False)
df = build_features(ctx["cots"], prod, jd)
OUT = ROOT / "outputs" / "model"; OUT.mkdir(parents=True, exist_ok=True)
print("행:", len(df), df["split"].value_counts().to_dict(), "/ 라벨 비율:", df[list(LABELS)].mean().round(3).to_dict())

## 1. 성능 (Validation 평가, Training 5겹 CV)

In [ ]:
met, coef, trees, F = fit_models(df)
print("피처", len(F), ":", F)
print(met.T.to_string())
met.to_csv(OUT / "metrics.csv", index=False, encoding="utf-8-sig")

## 2. 로지스틱 계수 (표준화)와 나무 중요도

In [ ]:
coef.to_csv(OUT / "coefficients.csv", index=False, encoding="utf-8-sig")
for y in LABELS:
    print("##", LABELS[y]); print(coef[coef["라벨"] == y].head(10).to_string(index=False))

## 3. 요인별 결함 비율

In [ ]:
rates = factor_rates(df)
rates.to_csv(OUT / "factor_rates.csv", index=False, encoding="utf-8-sig")
print(rates.to_string(index=False))

## 4. 의사결정나무 그림

In [ ]:
from src.plots import setup, SOURCE
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree
setup()
fig, axes = plt.subplots(2, 1, figsize=(22, 16))
for ax, y in zip(axes, LABELS):
    plot_tree(trees[y], feature_names=F, class_names=["결함 아님", "결함"], filled=True, impurity=False, proportion=True,
              rounded=True, fontsize=8, ax=ax)
    ax.set_title(f"{LABELS[y]} (깊이 3, min_samples_leaf 50, class_weight balanced, Training 학습)", fontsize=12)
fig.suptitle("07 결함 요인 모델: 의사결정나무", fontsize=14, fontweight="bold", x=0.02, ha="left")
fig.text(0.02, 0.005, SOURCE, fontsize=9, color="#666666", ha="left")
fig.tight_layout(rect=(0, 0.02, 1, 0.97))
fig.savefig(OUT / "tree.png", dpi=110); plt.close(fig)
print("저장:", OUT / "tree.png")

## 5. scorecard 저장

In [ ]:
from src.report import update_scorecard
rows = []
for _, r in met.iterrows():
    for k in ["기준선 Validation", "LR AUC", "LR 평균 정밀도", "LR Training 5겹 CV AUC", "DT AUC", "DT 평균 정밀도"]:
        rows.append({"지표": f"{r['설명']}: {k}", "Training": r["기준선 Training"] if k.startswith("기준선") else "",
                     "Validation": r[k] if not k.startswith("LR Training") else "", "전체": r[k] if k.startswith("LR Training") else "",
                     "단위": "", "출처": "07_결함요인모델", "근거 결정": "D66", "기준일": "2026-11-24"})
update_scorecard(rows, "결함 요인 모델 (07, D66)")
print("scorecard 저장:", len(rows), "행")

---
데이터: AI Hub(한국지능정보사회진흥원) 사업 결과